## Calcuating cost-savings based on derate model predictions

### Model test of interest: Neural Network
* Apply neural network and evaluate using kfold cross validation
* Correct for imbalanced dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.preprocessing import StandardScaler, OneHotEncoder, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.inspection import PartialDependenceDisplay

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

import warnings
warnings.filterwarnings("ignore")

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

### Define feature and target columns

In [2]:
## target columns
target_var = ['derate_2_6_hr',
              'derate_6_12_hr',
              'derate_12_24_hr', 
             'total_derates_2_hr', 
             'total_derates_6_hr', 
            'total_derates_12_hr', 
             'total_derates_24_hr']

## Predictor columns
# predict_var= ['year',
#            'month',
#            'weekday',
#               'hour',
 predict_var=['DistanceLtd',
              'EngineTimeLtd',
              'FuelLtd',
              'activeTransitionCount',
              'TurboBoostPressure',
              'AcceleratorPedal',
              'BarometricPressure',
              'CruiseControlSetSpeed',
              'EngineCoolantTemperature',
              'EngineLoad',
              'EngineOilPressure',
              'EngineOilTemperature',
              'EngineRpm',
              'FuelLevel',
              'FuelRate',
              'FuelTemperature',
              'IntakeManifoldTemperature',
              'Speed',
              'SwitchedBatteryVoltage',
              'Throttle',
              'active',
              'CruiseControlActive',
              'IgnStatus',
              'ParkingBrake',
              'spn',
              'fmi',
              'min_since_last_fault',
              'EquipmentID',
              'eventDescription',
              'LampStatus']

---

### cost prediction
* merge y_pred back with main dataset, and see if y_pred aligns with y_test and group by equipment and timewindow

In [3]:
#Neural network model application and view classification report
nn_pipe = Pipeline(
steps = [
    ('model', MLPClassifier(random_state=321, 
                            max_iter=5000, 
                            hidden_layer_sizes = (25,25,25), 
                            activation='relu', 
                           early_stopping=True, 
                           solver='adam'))
])

### Open up files for model: for k-fold evaluatuion, open up each test and validation dataset sequentially

In [4]:
#define final dataset to save results
model_evaluation_results= pd.DataFrame(columns=['target_var', 'fold_num', 'total_cost', 'total_pred_derate', 'total actual derate', 'f1-score'])


k_fold_num= [1, 2]
##iteration 1: open each csv file for each fold 
for fold in k_fold_num:
    fold_num = fold
    print(f"Applying k-fold number {fold}")
    
    #open train dataset csv file
    train_data= pd.read_csv(f'../processed_data/train_set_{fold}.csv')
    #open validation dataset csv file 
    val_data= pd.read_csv(f'../processed_data/val_set_{fold}.csv')

    #Iterate 2: apply model to each of the 7 possible target columns 
    for col_num in range(len(target_var)):
        #Select ONE target variable 
        target_var_opt = target_var[col_num]
        ##change the target columns to binary based on the target column selected
        train_data[train_data[target_var_opt]!= 0]=1
        
        #define columns to drop from X dataframes 
        drop_col= target_var + ['EquipmentID', 'EventTimeStamp']
        
        #Prepare train and test variables from the datasets 
        X_train = train_data.drop(columns=drop_col)
        #save model and event time separately 
        X_train_modeltime= train_data[['EquipmentID', 'EventTimeStamp']]
        #save target column as y_train
        y_train= train_data[target_var_opt].astype(int)
        
        #Prepare train and test variables from the datasets 
        X_test = val_data.drop(columns=drop_col)
        #save model and event time separately 
        X_val_modeltime= val_data[['EquipmentID', 'EventTimeStamp']]
        #save target column as y_train
        y_test= val_data[target_var_opt].astype(int)

        #Use SMOTE to over sample the minroity class and under sample the majority class 
        # Apply SMOTE only to training--> OVERSAMPLE minority class
        smote = SMOTE(random_state=321, sampling_strategy=0.1)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
        
        nn_pipe.fit(X_resampled, y_resampled)
        y_pred = nn_pipe.predict(X_test)
        f1_score_eval = classification_report(y_test, y_pred, output_dict=True)['macro avg']['f1-score']

        #calculate cost savings and other metrics
        print(f"calculating model performance for {target_var_opt} on {fold_num}")
        ### save it as a new unqiue dataset name 
        model_calc= X_test
        model_calc['pred_derate']= y_pred
        model_calc['actual_derate']= y_test
        model_calc['EquipmentID'] = X_val_modeltime['EquipmentID']
        model_calc['EventTimeStamp'] = X_val_modeltime['EventTimeStamp']

        ## group by equipment ID and calculate cost and model output metrics 

        model_calc['date']= pd.to_datetime(model_calc['EventTimeStamp']).dt.date
        model_cost_total= model_calc.groupby(['EquipmentID', 'date']).agg({'actual_derate': 'max', 'pred_derate': 'max'}).sort_values('actual_derate', ascending=False)
        
        #create a for loop to calculate cost based on formulas
        total_cost_list = []
        for row in range(len(model_cost_total)):
            eval_row = model_cost_total.iloc[row]
            if eval_row['actual_derate'] == 0 & eval_row['pred_derate'] == 1:
                total_cost_list.append(-500)
            elif eval_row['actual_derate'] == 1 & eval_row['pred_derate'] == 1:
                total_cost_list.append(4000)
            else:
                total_cost_list.append(0)
        
        ## add the final calculation list into the main dataframe 
        model_cost_total['total_cost']= total_cost_list

        ### save cost to final dataframe 
        model_pf = [target_var_opt, 
                    int(fold_num), 
                    model_cost_total['total_cost'].sum(), 
                    model_cost_total['pred_derate'].sum(), 
                    model_cost_total['actual_derate'].sum(), 
                   f1_score_eval]

        model_evaluation_results.loc[len(model_evaluation_results)] = model_pf
        print(f"{target_var_opt} on {fold_num} saved to final dataset")

Applying k-fold number 1
calculating model performance for derate_2_6_hr on 1
derate_2_6_hr on 1 saved to final dataset
calculating model performance for derate_6_12_hr on 1
derate_6_12_hr on 1 saved to final dataset
calculating model performance for derate_12_24_hr on 1
derate_12_24_hr on 1 saved to final dataset
calculating model performance for total_derates_2_hr on 1
total_derates_2_hr on 1 saved to final dataset
calculating model performance for total_derates_6_hr on 1
total_derates_6_hr on 1 saved to final dataset
calculating model performance for total_derates_12_hr on 1
total_derates_12_hr on 1 saved to final dataset
calculating model performance for total_derates_24_hr on 1
total_derates_24_hr on 1 saved to final dataset
Applying k-fold number 2
calculating model performance for derate_2_6_hr on 2
derate_2_6_hr on 2 saved to final dataset
calculating model performance for derate_6_12_hr on 2
derate_6_12_hr on 2 saved to final dataset
calculating model performance for derate_12

In [5]:
#view final dataframe
model_evaluation_results

,target_var,fold_num,total_cost,total_pred_derate,total actual derate,f1-score
0,derate_2_6_hr,1,0,0,55,0.499769
1,derate_6_12_hr,1,0,0,45,0.499807
2,derate_12_24_hr,1,0,0,107,0.499523
3,total_derates_2_hr,1,0,0,97,0.333163
4,total_derates_6_hr,1,0,0,134,0.249769
5,total_derates_12_hr,1,0,0,157,0.249696
6,total_derates_24_hr,1,0,0,245,0.199591
7,derate_2_6_hr,2,0,0,41,0.499883
8,derate_6_12_hr,2,0,0,16,0.499952
9,derate_12_24_hr,2,0,0,40,0.499801


---

In [35]:
#Select ONE target variable 
target_var_opt = target_var[0]
##change the target columns to binary based on the target column selected
train_data[train_data[target_var_opt]!= 0]=1

#define columns to drop from X dataframes 
drop_col= target_var + ['EquipmentID', 
                        'EventTimeStamp'] + ['year',
                                             'month',
                                             'weekday',
                                             'hour']

#Prepare train and test variables from the datasets 

In [37]:
X_train = train_data#.drop(columns=drop_col)
#save model and event time separately 
X_train_modeltime= train_data[['EquipmentID', 'EventTimeStamp']]
#save target column as y_train
y_train= train_data[target_var_opt].astype(int)

#Prepare train and test variables from the datasets 
X_test = val_data#.drop(columns=drop_col)
#save model and event time separately 
X_val_modeltime= val_data[['EquipmentID', 'EventTimeStamp']]
#save target column as y_train
y_test= val_data[target_var_opt].astype(int)

#Use SMOTE to over sample the minroity class and under sample the majority class 
# Apply SMOTE only to training--> OVERSAMPLE minority class
smote = SMOTE(random_state=321, sampling_strategy=0.1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

nn_pipe.fit(X_resampled, y_resampled)
y_pred = nn_pipe.predict(X_test)
f1_score_eval = classification_report(y_test, y_pred, output_dict=True)['macro avg']['f1-score']
y_pred_prob = nn_pipe.predict_proba(X_test)

ValueError: could not convert string to float: 'R1762'

In [29]:
y_pred_prob = nn_pipe.predict_proba(X_resampled)

In [31]:
print(confusion_matrix(y_resampled, y_pred_prob[:,1]>0.5)) #, output_dict=True)

[[375714      0]
 [     0  37571]]


In [26]:
y_train.sum()

1828

In [27]:
y_resampled.sum()

37571

In [15]:
y_pred_prob

array([[0.90766051, 0.09233949],
       [0.91956502, 0.08043498],
       [0.9403967 , 0.0596033 ],
       ...,
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ]])

In [ ]:
# #calculate cost savings and other metrics
# print(f"calculating model performance for {target_var_opt} on {fold_num}")
# ### save it as a new unqiue dataset name 
# model_calc= X_test
# model_calc['pred_derate']= y_pred
# model_calc['actual_derate']= y_test
# model_calc['EquipmentID'] = X_val_modeltime['EquipmentID']
# model_calc['EventTimeStamp'] = X_val_modeltime['EventTimeStamp']

# ## group by equipment ID and calculate cost and model output metrics 

# model_calc['date']= pd.to_datetime(model_calc['EventTimeStamp']).dt.date
# model_cost_total= model_calc.groupby(['EquipmentID', 'date']).agg({'actual_derate': 'max', 'pred_derate': 'max'}).sort_values('actual_derate', ascending=False)

# #create a for loop to calculate cost based on formulas
# total_cost_list = []
# for row in range(len(model_cost_total)):
#     eval_row = model_cost_total.iloc[row]
#     if eval_row['actual_derate'] == 0 & eval_row['pred_derate'] == 1:
#         total_cost_list.append(-500)
#     elif eval_row['actual_derate'] == 1 & eval_row['pred_derate'] == 1:
#         total_cost_list.append(4000)
#     else:
#         total_cost_list.append(0)

# ## add the final calculation list into the main dataframe 
# model_cost_total['total_cost']= total_cost_list

# ### save cost to final dataframe 
# model_pf = [target_var_opt, 
#             int(fold_num), 
#             model_cost_total['total_cost'].sum(), 
#             model_cost_total['pred_derate'].sum(), 
#             model_cost_total['actual_derate'].sum(), 
#            f1_score_eval]

# model_evaluation_results.loc[len(model_evaluation_results)] = model_pf
# print(f"{target_var_opt} on {fold_num} saved to final dataset")

In [13]:
nn_pipe.fit(X_resampled, y_resampled)
y_pred_prob = nn_pipe.predict_proba(X_test)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- EquipmentID
- EventTimeStamp
- actual_derate
- date
- pred_derate


-----

### Prepare datasets for train-test split, by removing any columns from the dataset that is NOT used in prediction, and saving the target variable column of interest in the Y value

In [6]:
# #Select ONE target variable 
# target_var_opt = target_var[0]
# ##change the target columns to binary based on the target column selected
# train_data[train_data[target_var_opt]!= 0]=1
# train_data[target_var_opt].value_counts()

# #define columns to drop from X dataframes 
# drop_col= target_var + ['EquipmentID', 'EventTimeStamp']

# #Prepare train and test variables from the datasets 
# X_train = train_data.drop(columns=drop_col)
# #save model and event time separately 
# X_train_modeltime= train_data[['EquipmentID', 'EventTimeStamp']]
# #save target column as y_train
# y_train= train_data[target_var_opt].astype(int)

# #Prepare train and test variables from the datasets 
# X_test = val_data.drop(columns=drop_col)
# #save model and event time separately 
# X_val_modeltime= val_data[['EquipmentID', 'EventTimeStamp']]
# #save target column as y_train
# y_test= val_data[target_var_opt].astype(int)

## Define the model pipeline, train it on the train dataset, and create predictions from the test dataset

In [7]:
# #Neural network model application and view classification report
# nn_pipe = Pipeline(
# steps = [
#     ('model', MLPClassifier(random_state=321, 
#                             max_iter=5000, 
#                             hidden_layer_sizes = (100,100), 
#                             activation='relu', 
#                            early_stopping=True, 
#                            solver='adam'))
# ])

# #Use SMOTE to over sample the minroity class and under sample the majority class 
# # Apply SMOTE only to training--> OVERSAMPLE minority class
# smote = SMOTE(random_state=321, sampling_strategy=0.1)
# X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# nn_pipe.fit(X_resampled, y_resampled)
# y_pred = nn_pipe.predict(X_test)

# f1_score_eval = classification_report(y_test, y_pred, output_dict=True)['macro avg']['f1-score']

### Add the predicited y_pred back to the full test dataset by adding it as a column, as well as equipment ID and event timestamp

In [8]:
# ### save it as a new unqiue dataset name 
# model_calc= X_test

# model_calc['pred_derate']= y_pred
# model_calc['actual_derate']= y_test
# model_calc['EquipmentID'] = X_val_modeltime['EquipmentID']
# model_calc['EventTimeStamp'] = X_val_modeltime['EventTimeStamp']

# #view dataframe to see column options
# model_calc.head(5)

---

# Calculate the company cost/savings based on model performance
Group by equipmentID and timestamp to get actual v predicted derate differnces

### For each truck, how many false positives did I have for each DAY? 
**For any truck-date combos:**
*  FALSE NEG- IF we failed to predict the derate (pred_derate=0) when an actual derate happened (actual_derate=1) -> no cost added because we already failed to catch it (so company is charged 4000 dollars regardless)
* FALSE POS- If we did predicit a derate (pred_derate = 1) when there was NOT a derate (actual_derate = 0) -> costs 500 dollars
* TRUE POS- IF we sucessfully predicted a derate (pred_derate = 1) when there WAS a derate (actuacl_derate = 1) -> SAVE 4000 dollars

In [9]:
# model_calc['date']= pd.to_datetime(model_calc['EventTimeStamp']).dt.date
# model_cost_total= model_calc.groupby(['EquipmentID', 'date']).agg({'actual_derate': 'max', 'pred_derate': 'max'}).sort_values('actual_derate', ascending=False)

# #create a for loop to calculate cost based on formulas
# total_cost_list = []
# for row in range(len(model_cost_total)):
#     eval_row = model_cost_total.iloc[row]
#     if eval_row['actual_derate'] == 0 & eval_row['pred_derate'] == 1:
#         total_cost_list.append(-500)
#     elif eval_row['actual_derate'] == 1 & eval_row['pred_derate'] == 1:
#         total_cost_list.append(4000)
#     else:
#         total_cost_list.append(0)

# ## add the final calculation list into the main dataframe 
# model_cost_total['total_cost']= total_cost_list

# #view and sort by total_cost per truck event 
# #model_cost_total.sort_values(['total_cost'], ascending=True)
# model_cost_total

# model_pf = [target_var_opt, 
#             int(fold_num), 
#             model_cost_total['total_cost'].sum(), 
#             model_cost_total['pred_derate'].sum(), 
#             model_cost_total['actual_derate'].sum(), 
#             0]

# model_evaluation_results= pd.DataFrame(columns=['target_var', 'fold_num', 'total_cost', 'total_pred_derate', 'total actual derate', 'f1-score'])
# model_evaluation_results.loc[len(model_evaluation_results)] = model_pf
# model_evaluation_results

## Calculate the total cost by adding up all the 'total_cost' columns 

In [10]:
# print("total cost: ", model_cost_total['total_cost'].sum())
# print("total predicted derates:", model_cost_total['pred_derate'].sum())
# print("total actual derates:", model_cost_total['actual_derate'].sum())

# #save model performance in a dictionary
# # #model_pf = dict({'target_var': target_var_opt, 
# #       'fold':int(fold_num), 
# #       'total_cost':model_cost_total['total_cost'].sum(), 
# #       'total_pred_derate':model_cost_total['pred_derate'].sum(), 
# #       'total actual derate':model_cost_total['actual_derate'].sum()})

# model_pf = [target_var_opt, 
#             int(fold_num), 
#             model_cost_total['total_cost'].sum(), 
#             model_cost_total['pred_derate'].sum(), 
#             model_cost_total['actual_derate'].sum(), 
#             0]

# model_evaluation_results= pd.DataFrame(columns=['target_var', 'fold_num', 'total_cost', 'total_pred_derate', 'total actual derate', 'f1-score'])
# model_evaluation_results.loc[len(model_evaluation_results)] = model_pf
# model_evaluation_results

---

In [72]:
target_var

['derate_2_6_hr',
 'derate_6_12_hr',
 'derate_12_24_hr',
 'total_derates_2_hr',
 'total_derates_6_hr',
 'total_derates_12_hr',
 'total_derates_24_hr']

In [113]:
#Neural network model application and view classification report
nn_pipe = Pipeline(
steps = [
    ('model', MLPClassifier(random_state=321, 
                            max_iter=5000, 
                            hidden_layer_sizes = (25,25,25), 
                            activation='tanh', 
                           early_stopping=True, 
                           solver='adam'))
])

In [131]:
# k_fold_num= [1]
# ##iteration 1: open each csv file for each fold 
# for fold in k_fold_num:
#     fold_num = fold
#     print(f"Applying k-fold number {fold}")

#open train dataset csv file
train_data= pd.read_csv(f'../processed_data/train_set_2.csv')
#open validation dataset csv file 
val_data= pd.read_csv(f'../processed_data/train_set_2.csv')

#Iterate 2: apply model to each of the 7 possible target columns 
#for col_num in range(len(target_var)):
#Select ONE target variable 
target_var_opt = target_var[0]
##change the target columns to binary based on the target column selected
#train_data[train_data[target_var_opt]!= 0]=1

drop_col= target_var + ['EquipmentID', 'EventTimeStamp', 'month_sin__month','month_cos__month','weekday_sin__weekday','weekday_cos__weekday','hour_sin__hour','hour_cos__hour', 'Unnamed: 0', 'year']
#Prepare train and test variables from the datasets 
X_train = train_data.drop(columns=drop_col)
#save model and event time separately 
X_train_modeltime= train_data[['EquipmentID', 'EventTimeStamp']]
#save target column as y_train
y_train= train_data[target_var_opt].astype(int)

#Prepare train and test variables from the datasets 
X_test = val_data.drop(columns=drop_col)
#save model and event time separately 
X_val_modeltime= val_data[['EquipmentID', 'EventTimeStamp']]
#save target column as y_train
y_test= val_data[target_var_opt].astype(int)

#Use SMOTE to over sample the minroity class and under sample the majority class 
# Apply SMOTE only to training--> OVERSAMPLE minority class
smote = SMOTE(random_state=321, sampling_strategy=0.1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)



In [141]:
nn_pipe.fit(X_train, y_train)

Pipeline(steps=[('model',
                 MLPClassifier(activation='tanh', early_stopping=True,
                               hidden_layer_sizes=(25, 25, 25), max_iter=5000,
                               random_state=321))])

In [143]:
X_train.columns.to_list()

['min_since_last_fault',
 'DistanceLtd',
 'EngineTimeLtd',
 'FuelLtd',
 'activeTransitionCount',
 'TurboBoostPressure',
 'AcceleratorPedal',
 'BarometricPressure',
 'CruiseControlSetSpeed',
 'EngineCoolantTemperature',
 'EngineLoad',
 'EngineOilPressure',
 'EngineOilTemperature',
 'EngineRpm',
 'FuelLevel',
 'FuelRate',
 'FuelTemperature',
 'IntakeManifoldTemperature',
 'Speed',
 'SwitchedBatteryVoltage',
 'Throttle',
 'LampStatus_0',
 'LampStatus_1023',
 'LampStatus_11',
 'LampStatus_1279',
 'LampStatus_16639',
 'LampStatus_16895',
 'LampStatus_17407',
 'LampStatus_17663',
 'LampStatus_18419',
 'LampStatus_18431',
 'LampStatus_2',
 'LampStatus_2035',
 'LampStatus_2047',
 'LampStatus_21503',
 'LampStatus_22527',
 'LampStatus_255',
 'LampStatus_4351',
 'LampStatus_50175',
 'LampStatus_511',
 'LampStatus_5119',
 'LampStatus_51199',
 'LampStatus_5375',
 'LampStatus_6143',
 'LampStatus_617',
 'LampStatus_62463',
 'LampStatus_63487',
 'LampStatus_65535',
 'LampStatus_9',
 'spn_0.0',
 'spn_1

In [142]:
y_pred_prob = nn_pipe.predict_proba(X_train)#X_test[X_train.columns])
print(confusion_matrix(y_train, y_pred_prob[:,1]>0.5)) #, output_dict=True)

[[188568      0]
 [   203      0]]


In [125]:
from sklearn.inspection import permutation_importance

In [126]:
# X_resampled, y_resampled

In [127]:
print(confusion_matrix(y_train.sample(n=10000, random_state=321), nn_pipe.predict(X_train.sample(n=10000, random_state=321))))

[[9988    0]
 [  12    0]]


In [103]:
perm_mean = permutation_importance(nn_pipe, 
                                   X_train.sample(n=10000, random_state=321), 
                                   y_train.sample(n=10000, random_state=321),
                                   n_repeats=10,
                                random_state=321, 
                                   scoring='f1')['importances_mean']
col_feature = X_train.columns.to_list()

pd.DataFrame({
    'variable': col_feature,
    'importance':perm_mean
}).sort_values('importance', ascending = True)

,variable,importance
0,min_since_last_fault,0.0
254,spn_5491.0,0.0
253,spn_54478.0,0.0
252,spn_5444.0,0.0
251,spn_5443.0,0.0
...,...,...
120,spn_236.0,0.0
119,spn_235.0,0.0
118,spn_2023.0,0.0
127,spn_256.0,0.0


In [ ]:
#Neural network model application and view classification report
nn_pipe = Pipeline(
steps = [
    ('model', MLPClassifier(random_state=321, 
                            max_iter=5000, 
                            hidden_layer_sizes = (25,25,25), 
                            activation='relu', 
                           early_stopping=True, 
                           solver='adam'))
])

In [104]:
pd.DataFrame({
    'variable': col_feature,
    'importance':perm_mean
}).sort_values('importance', ascending = False)

,variable,importance
0,min_since_last_fault,0.0
247,spn_53958.0,0.0
256,spn_558.0,0.0
255,spn_5571.0,0.0
254,spn_5491.0,0.0
...,...,...
123,spn_247.0,0.0
122,spn_245.0,0.0
121,spn_237.0,0.0
120,spn_236.0,0.0


In [107]:
perm_mean

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [100]:
pd.DataFrame({
    'variable': col_feature,
    'importance':perm_mean
}).sort_values('importance', ascending = True)

,variable,importance
0,min_since_last_fault,0.0
254,spn_5491.0,0.0
253,spn_54478.0,0.0
252,spn_5444.0,0.0
251,spn_5443.0,0.0
...,...,...
120,spn_236.0,0.0
119,spn_235.0,0.0
118,spn_2023.0,0.0
127,spn_256.0,0.0
